# Day 3: Assignment — Production-Ready RAG System

## Overview

Build a complete, documented RAG system ready for production handoff. This assignment extends your independent lab work with deeper evaluation (RAG triad), comprehensive error analysis, and a full RAG playbook.

### Grading Summary
| # | Deliverable | Points |
|---|-------------|--------|
| 1 | Knowledge Base Design | 15 |
| 2 | RAG System Prompt | 25 |
| 3 | RAG Outputs (15+ questions) | — |
| 4 | Golden Q&A Set (15+ items) | — |
| 5 | RAG Triad Metrics | 20 |
| 6 | Error Analysis | 25 |
| 7 | RAG Playbook | 15 |
| | **Total** | **100** |

---
## Setup

In [ ]:
!pip install -q -U google-genai

In [ ]:
import os
import json
import re
from typing import Optional, List, Tuple
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field

# Google GenAI
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except ImportError:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
    if not GEMINI_API_KEY:
        raise ValueError("GEMINI_API_KEY not found. Please set it in Colab Secrets or environment.")

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_ID = "gemini-2.5-flash-lite"
EMBEDDING_MODEL = "gemini-embedding-001"

print(f"Using model: {MODEL_ID}")
print(f"Using embedding: {EMBEDDING_MODEL}")

In [ ]:
# ── Infrastructure: API, Logging, and Core Functions ─────────────────────────

PROMPT_LOG = []  # Track all API calls

def _now():
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

def generate(prompt, label="default", temperature=0.3):
    """Generate text using the model."""
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config={"temperature": temperature, "max_output_tokens": 1000},
    )
    text = response.text
    PROMPT_LOG.append({
        "timestamp": _now(),
        "type": "generate",
        "label": label,
        "prompt_length": len(prompt),
        "response_length": len(text),
    })
    return text

def generate_structured(prompt, response_model: BaseModel, label="default", temperature=0.3):
    """Generate structured JSON response."""
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": response_model,
            "temperature": temperature,
        },
    )
    json_str = response.text
    PROMPT_LOG.append({
        "timestamp": _now(),
        "type": "generate_structured",
        "label": label,
        "prompt_length": len(prompt),
        "response_length": len(json_str),
    })
    return response_model.model_validate_json(json_str)

def embed_texts(texts, task_type="RETRIEVAL_DOCUMENT"):
    """Embed one or more texts using Gemini Embeddings API."""
    if isinstance(texts, str):
        texts = [texts]
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(task_type=task_type),
    )
    return [np.array(e.values) for e in response.embeddings]

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def chunk_sentences(text, max_words=80, overlap_words=20):
    """
    Split text into chunks at sentence boundaries.
    max_words: target chunk size
    overlap_words: overlap between chunks
    """
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    
    chunks = []
    current_chunk = []
    current_words = 0
    
    for sent in sentences:
        sent_words = len(sent.split())
        if current_words + sent_words > max_words and current_chunk:
            chunks.append(' '.join(current_chunk))
            # Overlap: keep last few sentences
            overlap_sents = []
            overlap_count = 0
            for s in reversed(current_chunk):
                overlap_count += len(s.split())
                if overlap_count > overlap_words:
                    break
                overlap_sents.insert(0, s)
            current_chunk = overlap_sents
            current_words = sum(len(s.split()) for s in current_chunk)
        
        current_chunk.append(sent)
        current_words += sent_words
    
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    
    return chunks

def search(query, chunks, chunk_embeddings, top_k=3):
    """
    Retrieve top-k relevant chunks using cosine similarity.
    Returns list of (index, score, text) tuples.
    """
    query_emb = embed_texts(query, task_type="RETRIEVAL_QUERY")[0]
    scores = []
    for i, doc_emb in enumerate(chunk_embeddings):
        sim = cosine_similarity(query_emb, doc_emb)
        scores.append((i, sim, chunks[i]))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

def rag_query(query, chunks, chunk_embeddings, top_k=3, system_prompt=""):
    """
    Execute a RAG query: retrieve context, then generate answer.
    Returns (answer, retrieved_results).
    """
    # Retrieve
    retrieved = search(query, chunks, chunk_embeddings, top_k=top_k)
    context = "\n\n".join(f"[Source {i}]: {text}" for i, (_, _, text) in enumerate(retrieved))
    
    # Generate
    rag_prompt = f"""{system_prompt}

Context:
{context}

Question: {query}

Answer:"""
    answer = generate(rag_prompt, label="rag_query")
    return answer, retrieved

# ── Hybrid Retrieval (optional, for bonus points) ────────
from sklearn.feature_extraction.text import TfidfVectorizer

def build_sparse_index(chunks):
    """Build TF-IDF sparse index for keyword retrieval."""
    vectorizer = TfidfVectorizer(stop_words='english')
    matrix = vectorizer.fit_transform(chunks)
    return vectorizer, matrix

def hybrid_search(query, doc_embeddings, chunks, vectorizer, tfidf_matrix,
                  top_k=5, semantic_weight=0.6, keyword_weight=0.4):
    """Combine semantic and keyword search with weighted scoring."""
    from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine
    # Semantic scores
    query_emb = embed_texts(query, task_type="RETRIEVAL_QUERY")[0]
    sem_scores = [cosine_similarity(query_emb, emb) for emb in doc_embeddings]
    # Sparse scores
    query_vec = vectorizer.transform([query])
    sparse_scores = sklearn_cosine(query_vec, tfidf_matrix).ravel()
    # Combine
    combined = []
    for i in range(len(chunks)):
        combo = semantic_weight * sem_scores[i] + keyword_weight * float(sparse_scores[i])
        combined.append((i, combo, chunks[i]))
    combined.sort(key=lambda x: x[1], reverse=True)
    return combined[:top_k]

def search_with_filter(query, doc_embeddings, chunks, chunk_sources, top_k=3,
                       filter_doc=None):
    """Semantic search with optional document source filter."""
    query_emb = embed_texts(query, task_type="RETRIEVAL_QUERY")[0]
    scores = []
    for i, doc_emb in enumerate(doc_embeddings):
        if filter_doc is not None and chunk_sources[i] != filter_doc:
            continue
        sim = cosine_similarity(query_emb, doc_emb)
        scores.append((i, sim, chunks[i]))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

# ── Retrieval Metrics ─────────────────────────────────────

def precision_recall_at_k(retrieved_indices, expected_indices, k):
    """Compute Precision@k and Recall@k."""
    retrieved_set = set(retrieved_indices[:k])
    expected_set = set(expected_indices)
    if not expected_set:
        return None, None
    hits = retrieved_set & expected_set
    precision = len(hits) / k if k > 0 else 0
    recall = len(hits) / len(expected_set)
    return precision, recall

print("All infrastructure ready (includes hybrid retrieval, metadata filtering, and retrieval metrics).")

In [ ]:
# ── RAG Triad Evaluation ────────────────────────────────────────────────────

class RAGTriadScore(BaseModel):
    """RAG triad evaluation for a single question."""
    context_relevance: int = Field(description="1-5: Are the retrieved chunks relevant to the question?")
    groundedness: int = Field(description="1-5: Is the answer supported by the retrieved chunks?")
    answer_relevance: int = Field(description="1-5: Does the answer address the question asked?")
    explanation: str = Field(description="Brief explanation of all three scores")

def evaluate_rag_triad(question, answer, retrieved_chunks, label="triad"):
    """Score a single RAG output on all three triad dimensions."""
    chunks_text = "\n\n".join(f"[Chunk {i+1}]: {c}" for i, c in enumerate(retrieved_chunks))
    eval_prompt = f"""Evaluate this RAG system output on three dimensions.

Question: {question}

Retrieved Context:
{chunks_text}

Generated Answer: {answer}

Score each dimension 1-5:
1. Context Relevance: Are the retrieved chunks relevant to answering this question?
2. Groundedness: Is the generated answer fully supported by the retrieved chunks? (5=fully grounded, 1=hallucinated)
3. Answer Relevance: Does the answer actually address what was asked?"""
    return generate_structured(eval_prompt, RAGTriadScore, label=label)

print("RAG Triad evaluation function loaded.")

---
## Part 1: Knowledge Base Design (15 points)

Design your knowledge base with 4-6 documents. You may reuse and expand documents from Lab 2 or create new ones.

**Document your decisions:**
- Domain and why you chose it
- Chunk size, overlap, and rationale
- Number of resulting chunks

In [ ]:
# ── TODO: Define your knowledge base ───────────────────────────────────────
# Include 4-6 documents with at least 1,000 total words

documents = [
    """TODO: Document 1 — [Title]
    
    [Your document content here, 150-300 words]
    """,
    """TODO: Document 2 — [Title]
    
    [Your document content here, 150-300 words]
    """,
    """TODO: Document 3 — [Title]
    
    [Your document content here, 150-300 words]
    """,
    """TODO: Document 4 — [Title]
    
    [Your document content here, 150-300 words]
    """,
    # Add more documents as needed (5-6 recommended)
]

total_words = sum(len(doc.split()) for doc in documents)
print(f"Documents: {len(documents)}")
print(f"Total words: {total_words}")
assert total_words >= 1000, f"Need at least 1,000 words, have {total_words}"

In [ ]:
# ── TODO: Configure and apply chunking ───────────────────
CHUNK_MAX_WORDS = 80     # TODO: Adjust based on your content
CHUNK_OVERLAP_WORDS = 20  # TODO: Adjust overlap

# Document your rationale:
CHUNKING_RATIONALE = """
TODO: Explain your chunking decisions:
- Why this chunk size?
- Why this overlap?
- Did you try other values? What happened?
"""

all_chunks = []
chunk_sources = []    # Track which document each chunk came from
chunk_metadata = []   # Rich metadata for each chunk
for doc_idx, doc in enumerate(documents):
    doc_title = doc.strip().split('\n')[0]  # First line as title
    chunks = chunk_sentences(doc, max_words=CHUNK_MAX_WORDS, overlap_words=CHUNK_OVERLAP_WORDS)
    for chunk_idx, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        chunk_sources.append(doc_idx)
        chunk_metadata.append({
            "chunk_id": f"DOC{doc_idx}::C{chunk_idx+1}",
            "doc_index": doc_idx,
            "doc_title": doc_title,
        })

chunk_embeddings = embed_texts(all_chunks, task_type="RETRIEVAL_DOCUMENT")

# Optional: build sparse index for hybrid retrieval
tfidf_vectorizer, tfidf_matrix = build_sparse_index(all_chunks)

print(f"Chunking: max_words={CHUNK_MAX_WORDS}, overlap={CHUNK_OVERLAP_WORDS}")
print(f"Total chunks: {len(all_chunks)}")
print(f"Dense index:  {len(chunk_embeddings)} embeddings")
print(f"Sparse index: {tfidf_matrix.shape[1]} vocabulary terms")
print(f"\nChunks per document:")
for i in range(len(documents)):
    count = chunk_sources.count(i)
    print(f"  Doc {i}: {count} chunks")
print(f"\n{CHUNKING_RATIONALE}")

---
## Part 2: RAG System Prompt (25 points)

Your prompt must include ALL 8 components: role, task, grounding rules, scope, format, refusal, examples, conflict handling.

In [ ]:
# ── TODO: Write your complete system prompt ─────────────────────────────────

FINAL_SYSTEM_PROMPT = """TODO: Write your production-quality system prompt.

Include ALL of these components:

1. ROLE: Who is this assistant?
2. TASK: What does it do?
3. GROUNDING RULES: How must it use the context?
4. SCOPE: What topics are in/out of scope?
5. FORMAT: Answer length, citation format, structure
6. REFUSAL: What to say when the answer is not in context
7. EXAMPLES: At least one example of a good answer
8. CONFLICT HANDLING: What to do when chunks disagree

"""

print("System prompt:")
print(FINAL_SYSTEM_PROMPT)
print(f"\nPrompt length: {len(FINAL_SYSTEM_PROMPT.split())} words")

---
## Part 3: Input Questions (15+)

In [ ]:
# ── TODO: Define 15+ questions to process ──────────────────────────────────
QUESTIONS = [
    "TODO: Question 1",
    "TODO: Question 2",
    "TODO: Question 3",
    "TODO: Question 4",
    "TODO: Question 5",
    "TODO: Question 6",
    "TODO: Question 7",
    "TODO: Question 8",
    "TODO: Question 9",
    "TODO: Question 10",
    "TODO: Question 11",
    "TODO: Question 12",
    "TODO: Question 13",
    "TODO: Question 14",
    "TODO: Question 15",
    # Add more as needed
]

assert len(QUESTIONS) >= 15, f"Need at least 15 questions, have {len(QUESTIONS)}"
print(f"Questions: {len(QUESTIONS)}")

---
## Part 4: Run RAG Pipeline

In [ ]:
# Run RAG on all questions
rag_outputs = []
for q in QUESTIONS:
    answer, retrieved = rag_query(
        q, all_chunks, chunk_embeddings,
        top_k=3, system_prompt=FINAL_SYSTEM_PROMPT
    )
    rag_outputs.append({
        "question": q,
        "answer": answer,
        "retrieved_chunks": [doc for _, _, doc in retrieved],
        "retrieval_scores": [score for _, score, _ in retrieved],
    })

print(f"Processed {len(rag_outputs)} questions\n")
for i, out in enumerate(rag_outputs[:3]):
    print(f"Q{i+1}: {out['question']}")
    print(f"A: {out['answer'][:100]}...\n")

In [ ]:
# Export RAG outputs
with open("day3_assignment_rag_outputs.json", "w") as f:
    json.dump(rag_outputs, f, indent=2, default=str)
print("Exported to day3_assignment_rag_outputs.json")

---
## Part 5: Golden Q&A Set (15+ items)

Create your golden set with expected keywords. Include easy, medium, hard, and refusal questions.

In [ ]:
# ── TODO: Create your golden test set ────────────────────
GOLDEN_SET = [
    # Easy (6-7 items)
    {"id": "Q01", "question": "TODO", "expected_keywords": ["TODO"], "expected_chunks": [], "difficulty": "easy"},
    {"id": "Q02", "question": "TODO", "expected_keywords": ["TODO"], "expected_chunks": [], "difficulty": "easy"},
    {"id": "Q03", "question": "TODO", "expected_keywords": ["TODO"], "expected_chunks": [], "difficulty": "easy"},
    {"id": "Q04", "question": "TODO", "expected_keywords": ["TODO"], "expected_chunks": [], "difficulty": "easy"},
    {"id": "Q05", "question": "TODO", "expected_keywords": ["TODO"], "expected_chunks": [], "difficulty": "easy"},
    {"id": "Q06", "question": "TODO", "expected_keywords": ["TODO"], "expected_chunks": [], "difficulty": "easy"},
    # Medium (4-5 items)
    {"id": "Q07", "question": "TODO", "expected_keywords": ["TODO"], "expected_chunks": [], "difficulty": "medium"},
    {"id": "Q08", "question": "TODO", "expected_keywords": ["TODO"], "expected_chunks": [], "difficulty": "medium"},
    {"id": "Q09", "question": "TODO", "expected_keywords": ["TODO"], "expected_chunks": [], "difficulty": "medium"},
    {"id": "Q10", "question": "TODO", "expected_keywords": ["TODO"], "expected_chunks": [], "difficulty": "medium"},
    # Hard / Edge (2-3 items)
    {"id": "Q11", "question": "TODO", "expected_keywords": ["TODO"], "expected_chunks": [], "difficulty": "hard"},
    {"id": "Q12", "question": "TODO", "expected_keywords": ["TODO"], "expected_chunks": [], "difficulty": "hard"},
    # Refusal (2-3 items)
    {"id": "Q13", "question": "TODO", "expected_keywords": ["don't have", "not in", "no information", "outside"], "expected_chunks": [], "difficulty": "refusal"},
    {"id": "Q14", "question": "TODO", "expected_keywords": ["don't have", "not in", "no information", "outside"], "expected_chunks": [], "difficulty": "refusal"},
    {"id": "Q15", "question": "TODO", "expected_keywords": ["don't have", "not in", "no information", "outside"], "expected_chunks": [], "difficulty": "refusal"},
]

assert len(GOLDEN_SET) >= 15, f"Need at least 15 items, have {len(GOLDEN_SET)}"
print(f"Golden set: {len(GOLDEN_SET)} questions")
print("NOTE: Fill in expected_chunks with the chunk indices you expect to be retrieved.")
print("      Leave empty ([]) for refusal questions.\n")
for q in GOLDEN_SET:
    print(f"  [{q['difficulty']:7s}] {q['id']}: {q['question']}")

---
## Part 6: RAG Triad Metrics (20 points)

Evaluate EVERY golden set question on all three RAG triad dimensions.

In [ ]:
# Run RAG + triad evaluation + retrieval metrics on golden set
triad_results = []

for qa in GOLDEN_SET:
    # Run RAG
    answer, retrieved = rag_query(
        qa["question"], all_chunks, chunk_embeddings,
        top_k=3, system_prompt=FINAL_SYSTEM_PROMPT
    )
    retrieved_texts = [doc for _, _, doc in retrieved]
    retrieved_indices = [idx for idx, _, _ in retrieved]
    
    # Keyword check
    answer_lower = answer.lower()
    keyword_hit = any(kw.lower() in answer_lower for kw in qa["expected_keywords"])
    
    # RAG Triad evaluation
    triad = evaluate_rag_triad(qa["question"], answer, retrieved_texts, label=f"triad_{qa['id']}")
    
    # Precision@k / Recall@k (if expected_chunks provided)
    p_at_k, r_at_k = None, None
    if qa.get("expected_chunks"):
        p_at_k, r_at_k = precision_recall_at_k(retrieved_indices, qa["expected_chunks"], k=3)
    
    triad_results.append({
        "id": qa["id"],
        "question": qa["question"],
        "difficulty": qa["difficulty"],
        "answer": answer,
        "keyword_match": keyword_hit,
        "context_relevance": triad.context_relevance,
        "groundedness": triad.groundedness,
        "answer_relevance": triad.answer_relevance,
        "precision_at_3": p_at_k,
        "recall_at_3": r_at_k,
        "explanation": triad.explanation,
    })

triad_df = pd.DataFrame(triad_results)

In [ ]:
# Display RAG Triad + Retrieval Metrics
print("=" * 70)
print("RAG TRIAD + RETRIEVAL EVALUATION RESULTS")
print("=" * 70)

keyword_acc = triad_df["keyword_match"].mean()
avg_context = triad_df["context_relevance"].mean()
avg_ground = triad_df["groundedness"].mean()
avg_relevance = triad_df["answer_relevance"].mean()

# Precision/Recall (only for non-refusal questions)
pr_df = triad_df.dropna(subset=["precision_at_3"])
avg_precision = pr_df["precision_at_3"].mean() if len(pr_df) > 0 else 0
avg_recall = pr_df["recall_at_3"].mean() if len(pr_df) > 0 else 0

print(f"\nOverall Metrics:")
print(f"  Keyword Accuracy:    {keyword_acc:.0%} ({triad_df['keyword_match'].sum()}/{len(triad_df)})")
print(f"  Context Relevance:   {avg_context:.1f}/5 {'✓' if avg_context >= 4.0 else '✗ (target: ≥ 4.0)'}")
print(f"  Groundedness:        {avg_ground:.1f}/5 {'✓' if avg_ground >= 4.0 else '✗ (target: ≥ 4.0)'}")
print(f"  Answer Relevance:    {avg_relevance:.1f}/5 {'✓' if avg_relevance >= 4.0 else '✗ (target: ≥ 4.0)'}")
if len(pr_df) > 0:
    print(f"  Avg Precision@3:     {avg_precision:.2f}")
    print(f"  Avg Recall@3:        {avg_recall:.2f}")

print(f"\nPer-Question Scores:")
display_cols = ["id", "difficulty", "keyword_match", "context_relevance",
                "groundedness", "answer_relevance"]
if len(pr_df) > 0:
    display_cols.extend(["precision_at_3", "recall_at_3"])
print(triad_df[display_cols].to_string(index=False))

# Flag questions below target
below_target = triad_df[
    (triad_df["context_relevance"] < 4) |
    (triad_df["groundedness"] < 4) |
    (triad_df["answer_relevance"] < 4)
]
if len(below_target) > 0:
    print(f"\nQuestions below target (< 4.0 on any dimension):")
    for _, row in below_target.iterrows():
        dims = []
        if row["context_relevance"] < 4: dims.append(f"Context={row['context_relevance']}")
        if row["groundedness"] < 4: dims.append(f"Ground={row['groundedness']}")
        if row["answer_relevance"] < 4: dims.append(f"Relevance={row['answer_relevance']}")
        print(f"  {row['id']}: {', '.join(dims)} — {row['explanation'][:80]}")

---
## Part 7: Error Analysis (25 points)

Write your error analysis below. Cover all three sections.

### A. Retrieval Failures

**Pattern 1:** [Name]
- **What went wrong:** [Description — wrong chunks retrieved or correct chunks ranked too low]
- **Example:** [Question ID, what was retrieved vs what should have been]
- **Root cause:** [Chunk size issue? Embedding limitation? Query phrasing?]

**Pattern 2:** [Name]
- **What went wrong:** [Description]
- **Example:** [Specific example]
- **Root cause:** [Analysis]

### B. Generation Failures

**Pattern 1:** [Name — e.g., hallucination, over-hedging, wrong scope]
- **What happened:** [Description]
- **Example:** [Question, context provided, and incorrect answer]
- **Prompt fix attempted:** [What you changed and whether it helped]

**Pattern 2:** [Name]
- **What happened:** [Description]
- **Example:** [Specific example]
- **Prompt fix attempted:** [Change and result]

### C. Refusal Testing

- **Correct refusals:** [How many out-of-scope questions were correctly refused?]
- **False refusals:** [Did the system refuse when it should have answered? Examples?]
- **Missed refusals:** [Did the system answer when it should have refused? Examples?]
- **Production improvements:** [What would you change for production refusal handling?]

---
## Part 8: RAG Playbook (15 points)

Complete ALL sections of the playbook template below.

---

# RAG PLAYBOOK: [Your System Name]

**Version:** 1.0
**Author:** [Your Name]
**Date:** [Date]
**Status:** Production Ready

## 1. Purpose
[TODO: What does this RAG system do? What problem does it solve? Who uses it?]

## 2. Knowledge Base
| Property | Value |
|----------|-------|
| Documents | [TODO] |
| Total words | [TODO] |
| Domain | [TODO] |
| Update frequency | [TODO] |

## 3. Chunking Strategy
| Parameter | Value | Rationale |
|-----------|-------|----------|
| Strategy | [TODO] | [TODO] |
| Chunk size | [TODO] | [TODO] |
| Overlap | [TODO] | [TODO] |
| Total chunks | [TODO] | — |

## 4. System Prompt
[TODO: Paste your complete final system prompt]

## 5. Model Settings
| Setting | Value | Rationale |
|---------|-------|----------|
| Generation model | gemini-2.5-flash-lite | [TODO] |
| Embedding model | gemini-embedding-001 | [TODO] |
| Temperature | [TODO] | [TODO] |
| top_k | [TODO] | [TODO] |

## 6. RAG Triad Performance
| Metric | Score | Target |
|--------|-------|--------|
| Context Relevance | [TODO] | ≥ 4.0 |
| Groundedness | [TODO] | ≥ 4.0 |
| Answer Relevance | [TODO] | ≥ 4.0 |
| Keyword Accuracy | [TODO] | ≥ 80% |

## 7. Known Limitations
1. [TODO: Limitation 1]
2. [TODO: Limitation 2]
3. [TODO: Limitation 3]

## 8. Deployment Considerations
- **Human review triggers:** [TODO]
- **Confidence threshold:** [TODO]
- **Monitoring:** [TODO]
- **Update process:** [TODO]

## 9. Operations
- **Knowledge base refresh:** [TODO]
- **Prompt versioning:** [TODO]
- **Incident response:** [TODO]

## 10. Version History
| Version | Changes | Context Rel. | Groundedness | Answer Rel. |
|---------|---------|-------------|-------------|-------------|
| 1.0 | Initial | [TODO] | [TODO] | [TODO] |

## 11. Handoff Checklist
- [ ] Knowledge base documented
- [ ] System prompt reviewed by team
- [ ] Test set of 15+ questions provided
- [ ] All RAG triad metrics meet targets
- [ ] Limitations acknowledged
- [ ] Monitoring plan in place

---
## Export & Submission

In [ ]:
# ── Export all files ──────────────────────────────────────────────────────────

# 1. Golden set
with open("day3_assignment_golden_set.json", "w") as f:
    json.dump(GOLDEN_SET, f, indent=2)
print("✓ Exported day3_assignment_golden_set.json")

# 2. Triad results
triad_export = triad_df.to_dict(orient="records")
with open("day3_assignment_triad_results.json", "w") as f:
    json.dump(triad_export, f, indent=2, default=str)
print("✓ Exported day3_assignment_triad_results.json")

# 3. Prompt log
if PROMPT_LOG:
    log_df = pd.DataFrame(PROMPT_LOG)
    log_df.to_csv("day3_assignment_prompt_log.csv", index=False)
    print(f"✓ Exported {len(PROMPT_LOG)} API calls to day3_assignment_prompt_log.csv")

# Submission checklist
print("\n" + "=" * 50)
print("SUBMISSION CHECKLIST")
print("=" * 50)
print(f"  [{'✓' if len(documents) >= 4 else '✗'}] 4-6 documents in knowledge base")
print(f"  [{'✓' if total_words >= 1000 else '✗'}] 1,000+ total words")
print(f"  [{'✓' if len(all_chunks) > 0 else '✗'}] Chunking strategy applied")
print(f"  [{'✓' if 'TODO' not in FINAL_SYSTEM_PROMPT else '✗'}] System prompt completed (no TODOs)")
print(f"  [{'✓' if len(QUESTIONS) >= 15 else '✗'}] 15+ questions processed")
print(f"  [{'✓' if len(GOLDEN_SET) >= 15 else '✗'}] 15+ golden set items")
print(f"  [{'✓' if len(triad_results) > 0 else '✗'}] RAG triad metrics computed")
print(f"  [ ] Error analysis completed (check markdown cells)")
print(f"  [ ] RAG playbook completed (check markdown cells)")

---
## Conclusion

Congratulations on completing your first production-ready RAG system!

In Day 4, you'll learn about **GenAI Agents** — systems that can call tools, reason over multiple steps, and use your RAG pipeline as one component of a larger autonomous workflow.

Your Day 3 RAG system becomes a building block: the agent will be able to *decide when to search your knowledge base*, formulate the right query, and integrate the retrieved answer into a multi-step plan.